In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# MatchScoutIA — TEST DE FAISABILITÉ : BASKET (enfants, petit terrain, filmé au téléphone) — 26/11/2026
# Ne construit rien : mesure ce que le SOCLE actuel sait faire sur cette vidéo, pour décider de la suite.
#   1. RF-DETR (même détecteur que le football) : joueurs et ballon visibles sur combien d'images ?
#   2. Gemini, image par image : phase de jeu, panier visible, score affiché, couleurs des équipes, caméra qui bouge ?
#   3. Gemini, sur des SÉQUENCES de 20 s : « un panier est-il marqué ? » (autour des paniers connus si vous en donnez
#      quelques-uns, et sur des séquences prises au hasard pour mesurer les fausses alertes)
# ENTRÉES : le dataset de la vidéo de basket ; mziparameter (clé Gemini). GPU conseillé (RF-DETR), pas obligatoire.
# COÛT ESTIMÉ : moins de 1 $ de Gemini (≈ 20 questions sur une image + ≈ 10 sur des séquences). Le coût réel s'affiche.
# À GARDER : /kaggle/working/test_basket.zip (images annotées + réponses de Gemini + bilan)
# ════════════════════════════════════════════════════════════════════════════════════════════
import glob, json, os, random, subprocess, sys, time, zipfile

# ── RÉGLAGES ───────────────────────────────────────────────────────────────────────────────────
VIDEO = "/kaggle/input/datasets/michaelziant/basketjeune/basketJeune.mp4"   # match de jeunes, règles adultes (31 min)
DEBUT_S, DUREE_S = 0, 1900 # TOUT le match
# (facultatif mais très utile) quelques paniers que VOUS avez vus dans cette portion, en heure de la vidéo :
# 26/11 : vérité donnée par l'utilisateur (heure vidéo → numéro et équipe du marqueur)
# 26/11 : vérité de l'utilisateur pour basketJeune.mp4 (heure vidéo → marqueur). 14:51 : « annulé je pense » → ni
# compté comme manqué, ni comme fausse alerte (INCERTAINS)
MARQUEURS = {"01:04": "31 vert", "02:27": "3 blanc", "07:39": "30 vert", "08:09": "32 vert", "11:59": "5 blanc",
             "12:49": "30 vert", "14:06": "5 blanc", "16:26": "23 vert", "16:41": "3 blanc", "18:44": "11 vert",
             "21:03": "14 blanc", "21:14": "31 vert", "21:58": "32 vert", "24:28": "11 vert", "24:44": "3 blanc",
             "25:32": "11 vert", "26:00": "11 vert", "27:30": "5 blanc", "28:05": "11 vert", "29:42": "3 blanc",
             "30:10": "3 blanc", "30:26": "13 vert"}
PANIERS_CONNUS = list(MARQUEURS)
INCERTAINS = []
# 26/11 : vérifiés par l'utilisateur — 14:51 panier ANNULÉ, 20:49 lancer franc RATÉ (le 2e essai les annonçait)
ANNULES_OU_RATES = ["14:51", "20:49"]
# 26/11 (7e essai) : fenêtre au cercle élargie (−1 → +4 s) ; contrôle de la REPRISE après chaque tir (remise en jeu de
# l'autre équipe = panier ; jeu qui continue = raté ; coup de sifflet / remise sur le côté = faute, panier annulé ?)
FENETRE_CERCLE = (-1.0, 4.0)
CONTROLE_REPRISE = True
# 26/11 : ESSAI CIBLÉ, presque gratuit. L'étape 1 (repérage des tirs) est relue dans le cache du run précédent (0 $ :
# ajouter gemini_cache_basket.jsonl du run précédent en ENTRÉE). Les étapes 2 et reprise ne sont faites QUE pour les
# tirs proches de ces instants : les 7 manqués, les 2 fausses alertes, et 3 paniers trouvés (pour vérifier qu'ils le
# restent). Vide = tous les tirs.
# 26/11 : MODE « DIFFÉRÉ » (Gemini Flex, −50 %, comme l'offre « sous 48 h ») : chaque question peut attendre plusieurs
# minutes chez Google → les questions sont posées EN PARALLÈLE (sinon le test durerait des heures)
FLEX = True
PARALLELE = 32
CIBLES = ["07:39", "08:09", "16:26", "16:41", "18:44", "24:28", "29:42", "14:51", "20:49", "02:27", "12:49", "25:32"]
N_SEQUENCES_HASARD = 5     # séquences sans panier connu, pour mesurer les fausses alertes
# 26/11 : BALAYAGE de toute la portion par séquences de 20 s qui se chevauchent (une toutes les 10 s) : c'est ce que
# ferait un vrai moteur basket ; mesure les paniers trouvés, les fausses alertes et le numéro du marqueur
BALAYAGE = True
SAUTER_ETAPES_1_2 = True   # déjà faites au 1er essai
# 26/11 (3e essai) : envoyer à Gemini un VRAI EXTRAIT VIDÉO de 20 s (FPS_VIDEO images/s) au lieu de 11 images espacées de
# 2 s : le ballon ne traverse le filet que pendant une fraction de seconde (00:10 et 03:17 manqués au 2e essai)
SEQUENCE_EN_VIDEO = False    # 3e essai (extraits vidéo) : 1/4 seulement — Gemini les a lus en trop peu de détails
FPS_VIDEO = 4
# 26/11 (4e essai) : images comme au 2e essai (2/4, 0 fausse alerte) mais UNE PAR SECONDE (au lieu d'une toutes les
# 2 s) et la RÈGLE DE LA REPRISE (après un panier, remise en jeu par l'autre équipe) dans la question
PAS_IMAGES_S = 1
# 26/11 (5e essai) : DEUX ÉTAPES. Les paniers de 00:10 et 03:17 sont BIEN visibles dans le filet (captures de
# l'utilisateur), mais le cercle ne fait que ~30 px sur une image réduite à 960 px → « rebondit sur le cercle ».
#   étape 1 : balayage toutes les 2 s (comme le 2e essai) → Gemini liste les TIRS (instant + position du panier visé) ;
#   étape 2 : pour chaque tir, images en PLEINE RÉSOLUTION, recadrées autour du panier, 5 par seconde de −1 s à +2 s
#             → « le ballon passe-t-il dans le filet ? » (ne dépend plus de la reprise du jeu)
DEUX_ETAPES = True
# ───────────────────────────────────────────────────────────────────────────────────────────────

REPO = "/kaggle/working/ProjectAIMika"
SORTIE = "/kaggle/working/test_basket"
os.makedirs(SORTIE, exist_ok=True)
if not os.path.isdir(REPO):
    os.system(f"git clone -q --depth 1 https://github.com/mikarocco-droid/ProjectAIMika.git {REPO}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rfdetr", "google-genai", "opencv-python-headless"], check=False)
os.chdir(REPO); sys.path.insert(0, REPO)
for l in open(next(iter(glob.glob("/kaggle/input/**/param.env", recursive=True))), encoding="utf-8"):
    if l.strip().startswith("GEMINI_API_KEY="):
        os.environ["GEMINI_API_KEY"] = l.split("=", 1)[1].strip().strip('"').strip("'")
os.environ["SCOUTIA_GEMINI_CACHE_OUT"] = f"{SORTIE}/gemini_cache_basket.jsonl"
if FLEX:
    os.environ["SCOUTIA_GEMINI_TIER"] = "flex"
import ai.gemini_validator as gv          # VERROU d'abord : modèle unique gemini-3.8-flash + cache + comptage des coûts
import cv2
import numpy as np

if not VIDEO:
    cands = [p for p in glob.glob("/kaggle/input/**/*.mp4", recursive=True) + glob.glob("/kaggle/input/**/*.MP4", recursive=True)
             + glob.glob("/kaggle/input/**/*.mov", recursive=True) + glob.glob("/kaggle/input/**/*.MOV", recursive=True)
             if "basket" in p.lower()]
    if not cands:
        raise SystemExit("❌ aucune vidéo dont le chemin contient « basket » : indiquez VIDEO = \"/kaggle/input/…\"")
    VIDEO = cands[0]
cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
duree = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fin = min(duree, DEBUT_S + DUREE_S)
print(f"VIDÉO : {VIDEO}\n  {w}x{h} | {fps:.1f} images/s | {duree / 60:.1f} min | portion testée {DEBUT_S}→{fin:.0f} s")
mmss = lambda t: f"{int(t) // 60:02d}:{int(t) % 60:02d}"
sec = lambda h_: int(h_.split(":")[0]) * 60 + int(h_.split(":")[1])


def image_a(t):
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ok, img = cap.read()
    return img if ok else None


# ── 1. RF-DETR : joueurs et ballon ─────────────────────────────────────────────────────────────
print("\n══ 1. DÉTECTION (RF-DETR, comme au football) ══")
if SAUTER_ETAPES_1_2:
    print("  (sautée : déjà faite)")
    instants = []
if not SAUTER_ETAPES_1_2:
    from rfdetr import RFDETRMedium
    modele = RFDETRMedium()
    try:
        modele.optimize_for_inference()
    except Exception:
        pass
    instants = [DEBUT_S + k * 10 for k in range(int((fin - DEBUT_S) // 10))]       # une image toutes les 10 s
n_pers, ballon, tailles = [], 0, []
for i, t in enumerate(instants):
    img = image_a(t)
    if img is None:
        continue
    dets = modele.predict(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), threshold=0.3)
    pers = [b for b, k in zip(dets.xyxy, dets.class_id) if int(k) == 1]       # 1 = personne (COCO, calibré au football)
    bal = [b for b, k in zip(dets.xyxy, dets.class_id) if int(k) == 37]       # 37 = ballon de sport
    n_pers.append(len(pers)); ballon += bool(bal)
    tailles += [float(b[3] - b[1]) / h for b in pers]
    if i % 6 == 0:                                                            # quelques images annotées à regarder
        for b in pers:
            cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 255, 0), 2)
        for b in bal:
            cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 3)
        cv2.imwrite(f"{SORTIE}/detection_{mmss(t).replace(':', 'm')}.jpg", img)
if n_pers:
    print(f"  {len(n_pers)} images | personnes par image : médiane {int(np.median(n_pers))} (min {min(n_pers)}, max {max(n_pers)})"
          f" | BALLON vu sur {100 * ballon / len(n_pers):.0f} % des images"
          f" | taille d'un joueur : {100 * np.median(tailles):.0f} % de la hauteur de l'image (football Veo ≈ 5 à 10 %)")

# ── 2. Gemini sur une image ───────────────────────────────────────────────────────────────────
print("\n══ 2. GEMINI, IMAGE PAR IMAGE ══")
from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
QUESTION_IMAGE = (
    "Youth basketball game filmed with a phone. Answer ONLY with JSON: {\"phase\": \"jeu|lancer franc|remise en jeu|"
    "arret|temps mort|hors match\", \"panier_visible\": true/false, \"les_deux_paniers_visibles\": true/false, "
    "\"score_affiche\": \"texte du tableau de score lisible ou ''\", \"equipe_A\": \"couleur maillot\", "
    "\"equipe_B\": \"couleur maillot\", \"numeros_lisibles\": nombre de numéros de maillot lisibles, "
    "\"cadrage\": \"large|serre|suit l'action\"}")
reponses_img = []
for t in instants[::3][:20]:
    img = image_a(t)
    if img is None:
        continue
    _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 85])
    try:
        r = client.models.generate_content(model="gemini-3.8-flash", contents=[
            QUESTION_IMAGE, types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")], config={"temperature": 0})
        txt = r.text or ""
        d = json.loads(txt[txt.index("{"):txt.rindex("}") + 1])
    except Exception as e:
        d = {"erreur": str(e)[:120]}
    d["t"] = mmss(t)
    reponses_img.append(d)
    print(f"  {mmss(t)} : {d}")

# ── 3. Gemini sur des séquences de 20 s : « un panier est-il marqué ? » ───────────────────────
print("\n══ 3. GEMINI, SÉQUENCES DE 20 s : PANIER MARQUÉ ? ══")
QUESTION_SEQ = (
    "These are consecutive frames (timestamps given) of a youth basketball game filmed with a phone (mini-basket, "
    "low hoops). Is a BASKET SCORED (ball going through the hoop) in this sequence? Decisive rule: AFTER A BASKET, play "
    "stops and the OTHER team restarts with a throw-in (inbound pass) from outside the court. If play simply CONTINUES "
    "after the shot (rebound), there was NO basket. If the referee whistles and play restarts from the SIDE line, it is "
    "a foul or the ball went out, NOT a basket. Other clues: the ball drops through the net, the scorers celebrate. "
    "Answer ONLY with JSON: {\"panier\": true/false, \"reprise\": \"remise en jeu par l'autre equipe|le jeu continue|"
    "coup de sifflet, remise sur le cote|pas de tir|incertain\", "
    "\"instant\": \"mm:ss or ''\", \"equipe\": \"maillot couleur or ''\", \"marqueur\": \"numéro du tireur ou ''\", "
    "\"passeur\": \"numéro du passeur décisif ou ''\", \"certitude\": \"haute|moyenne|faible\", "
    "\"preuve\": \"one short sentence\"}")
fenetres = [("connu", sec(p)) for p in PANIERS_CONNUS]
random.seed(1)
hasard = [t for t in range(DEBUT_S + 20, int(fin) - 20, 20) if all(abs(t - sec(p)) > 30 for p in PANIERS_CONNUS)]
fenetres += [("hasard", t) for t in random.sample(hasard, min(N_SEQUENCES_HASARD, len(hasard)))]
reponses_seq = []
if BALAYAGE:
    fenetres = [("balayage", t) for t in range(DEBUT_S + 10, int(fin) - 9, 10)]
QUESTION_TIRS = (
    "These are consecutive frames (timestamps given, every 2 seconds) of a youth basketball game filmed with a phone "
    "(mini-basket, low mobile hoops). List EVERY SHOT at a hoop (any attempt, made or missed). For each: the timestamp of "
    "the frame closest to the release or the ball at the rim, and the bounding box of the TARGETED HOOP (rim + net) on "
    "that frame as [ymin, xmin, ymax, xmax] scaled 0-1000. Answer ONLY with JSON: {\"tirs\": [{\"instant\": \"mm:ss\", "
    "\"panier_box\": [ymin, xmin, ymax, xmax], \"tireur\": \"numéro et couleur ou ''\"}]}")
QUESTION_CERCLE = (
    "These are close-up crops (full resolution, 5 per second, timestamps given) around a basketball hoop, during a shot "
    "in a youth game. Does the BALL GO THROUGH THE NET (basket scored)? A ball touching the rim and falling outside is NOT "
    "a basket. Answer ONLY with JSON: {\"panier\": true/false, \"instant\": \"mm:ss\", \"certitude\": "
    "\"haute|moyenne|faible\", \"preuve\": \"one short sentence\"}")


QUESTION_REPRISE = (
    "A shot at a hoop happened at {T} in a basketball game (youth players, adult rules). These frames come AFTER the shot "
    "(timestamps given). How does play continue? \"remise en jeu par l'autre equipe\" = the defending team restarts with "
    "a throw-in from behind the END line under that hoop (this happens after a made basket); \"le jeu continue\" = "
    "rebound and play goes on (missed shot); \"coup de sifflet, remise sur le cote\" = the referee stops play (foul, "
    "violation, out of bounds) and play restarts from the side or with free throws; \"lancers francs\" = free throws "
    "follow. Answer ONLY with JSON: {\"reprise\": \"remise en jeu par l'autre equipe|le jeu continue|coup de sifflet, "
    "remise sur le cote|lancers francs|incertain\", \"preuve\": \"one short sentence\"}")


def demander(contenu):
    r = client.models.generate_content(model="gemini-3.8-flash", contents=contenu, config={"temperature": 0})
    txt = r.text or ""
    return json.loads(txt[txt.index("{"):txt.rindex("}") + 1])


if DEUX_ETAPES:
    print("  étape 1 : repérage des tirs (images toutes les 2 s)")
    tirs = []
    from concurrent.futures import ThreadPoolExecutor
    import threading as _th
    _verrou_video = _th.Lock()                     # une seule lecture vidéo à la fois (OpenCV n'aime pas le parallèle)

    def _sequence_tirs(tc):
        contenu = [QUESTION_TIRS]
        for t in range(tc - 10, tc + 11, 2):
            with _verrou_video:
                img = image_a(t)
            if img is None:
                continue
            img = cv2.resize(img, (960, int(960 * img.shape[0] / img.shape[1])))
            _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
            contenu += [f"[{mmss(t)}]", types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")]
        try:
            return demander(contenu).get("tirs") or []
        except Exception as e:
            print(f"    séquence {mmss(tc)} : {e}")
            return []

    with ThreadPoolExecutor(max_workers=PARALLELE if FLEX else 4) as _pool:
        _listes = list(_pool.map(_sequence_tirs, list(range(DEBUT_S + 10, int(fin) - 9, 20))))
    for _l in _listes:
        for x in _l:
            if ":" in str(x.get("instant", "")) and len(x.get("panier_box") or []) == 4:
                t_ = sec(x["instant"])
                if all(abs(t_ - y["t"]) > 2 for y in tirs):
                    tirs.append({"t": t_, "box": x["panier_box"], "tireur": x.get("tireur", "")})
    print(f"  {len(tirs)} tir(s) repéré(s) : {[mmss(x['t']) + ' ' + str(x['tireur']) for x in sorted(tirs, key=lambda y: y['t'])]}")
    print("  étape 2 : vérification au cercle (pleine résolution, recadrée)")
    if CIBLES:
        tirs = [x for x in tirs if any(abs(x["t"] - sec(c)) <= 6 for c in CIBLES)]
        print(f"  ESSAI CIBLÉ : {len(tirs)} tir(s) vérifié(s) : {[mmss(x['t']) for x in sorted(tirs, key=lambda y: y['t'])]}")
    def _image_verrou(t):
        with _verrou_video:
            return image_a(t)

    def _verifier_tir(x):
        y0, x0, y1, x1 = [float(v) / 1000 for v in x["box"]]
        cy, cx = (y0 + y1) / 2 * h, (x0 + x1) / 2 * w
        demi = max(140, 2.2 * max((y1 - y0) * h, (x1 - x0) * w))          # cadre ≈ 4 à 5 fois le panier
        contenu = [QUESTION_CERCLE]
        for k in range(int(FENETRE_CERCLE[0] * 5), int(FENETRE_CERCLE[1] * 5) + 1):   # 5 images/s
            t = x["t"] + k * 0.2
            img = _image_verrou(t)
            if img is None:
                continue
            ya, yb = int(max(0, cy - demi)), int(min(h, cy + demi * 1.6))   # plus de place sous le cercle (filet)
            xa, xb = int(max(0, cx - demi)), int(min(w, cx + demi))
            crop = img[ya:yb, xa:xb]
            if crop.size == 0:
                continue
            _, jpg = cv2.imencode(".jpg", crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
            contenu += [f"[{mmss(t)}.{int(round((t % 1) * 10))}]", types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")]
            if k == 0:
                cv2.imwrite(f"{SORTIE}/cercle_{mmss(x['t']).replace(':', 'm')}.jpg", crop)
        try:
            d = demander(contenu)
        except Exception as e:
            d = {"erreur": str(e)[:120]}
        d.update({"instant": d.get("instant") or mmss(x["t"]), "marqueur": x["tireur"], "nature": "balayage",
                  "centre": mmss(x["t"])})
        if ":" in str(d.get("instant", "")):
            d["instant"] = str(d["instant"]).split(".")[0]
        if CONTROLE_REPRISE:
            contenu_r = [QUESTION_REPRISE.replace("{T}", mmss(x["t"]))]
            for t in range(int(x["t"]) + 1, int(x["t"]) + 14, 2):
                img = _image_verrou(t)
                if img is None:
                    continue
                img = cv2.resize(img, (640, int(640 * img.shape[0] / img.shape[1])))
                _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 75])
                contenu_r += [f"[{mmss(t)}]", types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")]
            try:
                rp = demander(contenu_r)
            except Exception as e:
                rp = {"reprise": "incertain", "preuve": str(e)[:80]}
            d["reprise"] = rp.get("reprise", "incertain")
            # DÉCISION : le cercle d'abord ; la reprise confirme, met en doute (faute → annulé ?) ou rattrape
            if d.get("panier") and "sifflet" in str(d["reprise"]):
                d["statut"] = "à vérifier (annulé ? coup de sifflet après le panier)"
            elif d.get("panier"):
                d["statut"] = "panier"
            elif "remise en jeu" in str(d["reprise"]):
                d["panier"] = True
                d["statut"] = "à vérifier (cercle : non ; reprise : remise en jeu de l'autre équipe)"
            else:
                d["statut"] = "non"
        return d

    with ThreadPoolExecutor(max_workers=PARALLELE if FLEX else 4) as _pool:
        _tirs_tries = sorted(tirs, key=lambda y: y["t"])
        _resultats = list(_pool.map(_verifier_tir, _tirs_tries))
    for x, d in zip(_tirs_tries, _resultats):
        reponses_seq.append(d)
        print(f"    tir {mmss(x['t'])} ({x['tireur']}) → cercle {d.get('panier')} | reprise : {d.get('reprise', '—')} "
              f"| {d.get('statut', '')} | {d.get('preuve', d.get('erreur', ''))}")
    fenetres = []                                                          # étape 3 (ancienne méthode) sautée
QUESTION_VIDEO = (
    "This is a 20-second clip of a youth basketball game filmed with a phone (mini-basket, low hoops). "
    "Is a BASKET SCORED (ball going through the hoop) in this clip? Decisive rule: AFTER A BASKET, play stops and "
    "the OTHER team restarts with a throw-in (inbound pass) from outside the court. If play simply CONTINUES after "
    "the shot (rebound, ball kept alive), there was NO basket. If the referee whistles and play restarts from the "
    "SIDE line, it is a foul or the ball went out, NOT a basket. Other clues: the ball drops through the net, the "
    "scorers celebrate. Answer ONLY with JSON: {\"panier\": true/false, "
    "\"instant_dans_extrait\": \"mm:ss from the START of this clip or ''\", \"equipe\": \"maillot couleur or ''\", "
    "\"marqueur\": \"numéro du tireur ou ''\", \"passeur\": \"numéro du passeur décisif ou ''\", "
    "\"reprise\": \"remise en jeu par l'autre equipe|le jeu continue|coup de sifflet, remise sur le cote|"
    "pas de tir|incertain\", \"certitude\": \"haute|moyenne|faible\", \"preuve\": \"one short sentence\"}")


def extrait_video(t0, duree=20):
    """Extrait de `duree` s à partir de t0, réduit (640 px, sans son) pour limiter le coût."""
    sortie = f"{SORTIE}/extrait_{int(t0)}.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(max(0, t0)), "-i", VIDEO, "-t", str(duree),
                    "-vf", "scale=640:-2", "-an", "-c:v", "libx264", "-crf", "30", "-preset", "veryfast", sortie], check=False)
    return open(sortie, "rb").read() if os.path.exists(sortie) else None


for nature, tc in fenetres:
    if SEQUENCE_EN_VIDEO:
        octets = extrait_video(tc - 10)
        if not octets:
            continue
        try:
            part = types.Part(inline_data=types.Blob(data=octets, mime_type="video/mp4"),
                              video_metadata=types.VideoMetadata(fps=FPS_VIDEO))
        except Exception:
            part = types.Part.from_bytes(data=octets, mime_type="video/mp4")
        try:
            r = client.models.generate_content(model="gemini-3.8-flash", contents=[part, QUESTION_VIDEO],
                                               config={"temperature": 0})
            txt = r.text or ""
            d = json.loads(txt[txt.index("{"):txt.rindex("}") + 1])
            rel = str(d.get("instant_dans_extrait") or "")
            if ":" in rel:
                d["instant"] = mmss(tc - 10 + sec(rel))
        except Exception as e:
            d = {"erreur": str(e)[:160]}
        d.update({"centre": mmss(tc), "nature": nature})
        reponses_seq.append(d)
        print(f"  {nature:6s} {mmss(tc)} → panier {d.get('panier')} {d.get('instant', '')} n° {d.get('marqueur', '')} "
              f"| reprise : {d.get('reprise', '?')} ({d.get('certitude', '')}) | {d.get('preuve', d.get('erreur', ''))}")
        continue
    contenu = [QUESTION_SEQ]
    for t in range(tc - 10, tc + 11, PAS_IMAGES_S):
        img = image_a(t)
        if img is None:
            continue
        img = cv2.resize(img, (960, int(960 * img.shape[0] / img.shape[1])))
        _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
        contenu += [f"[{mmss(t)}]", types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")]
    try:
        r = client.models.generate_content(model="gemini-3.8-flash", contents=contenu, config={"temperature": 0})
        txt = r.text or ""
        d = json.loads(txt[txt.index("{"):txt.rindex("}") + 1])
    except Exception as e:
        d = {"erreur": str(e)[:120]}
    d.update({"centre": mmss(tc), "nature": nature})
    reponses_seq.append(d)
    print(f"  {nature:6s} {mmss(tc)} → panier {d.get('panier')} {d.get('instant', '')} n° {d.get('marqueur', '')} "
          f"| reprise : {d.get('reprise', '?')} ({d.get('certitude', '')}) | {d.get('preuve', d.get('erreur', ''))}")

# ── BILAN ──────────────────────────────────────────────────────────────────────────────────────
print("\n══ BILAN ══")
if BALAYAGE:
    # les séquences se chevauchent : un même panier vu deux fois (à moins de 8 s) ne compte qu'une fois
    trouves = []
    for d in reponses_seq:
        if d.get("panier") and d.get("instant") and ":" in str(d["instant"]):
            try:
                t_ = sec(d["instant"])
            except Exception:
                continue
            if all(abs(t_ - x[0]) > 8 for x in trouves):
                trouves.append((t_, d))
    print(f"  {len(trouves)} panier(s) annoncé(s) :")
    import re as _re
    n_num_ok = n_num = 0
    for t_, d in sorted(trouves, key=lambda x: x[0]):
        vrai = next((p for p in PANIERS_CONNUS if abs(sec(p) - t_) <= 8), None)
        incertain = any(abs(sec(p) - t_) <= 8 for p in INCERTAINS)
        annule = next((p for p in ANNULES_OU_RATES if abs(sec(p) - t_) <= 8), None)
        etat = ("✅ vrai panier " + vrai + " : " + MARQUEURS.get(vrai, "")) if vrai else \
               ("❔ panier incertain (annulé ?)" if incertain else
                f"❌ FAUSSE ALERTE (vérifié : {'annulé' if annule == '14:51' else 'lancer franc raté'})" if annule else
                "❌ FAUSSE ALERTE")
        if str(d.get("statut", "")).startswith("à vérifier"):
            etat += f"  [{d['statut']}]"
        if vrai:
            n_vrai = _re.findall(r"\d+", MARQUEURS.get(vrai, ""))[:1]
            n_lu = _re.findall(r"\d+", str(d.get("marqueur") or ""))[:1]
            if n_lu:
                n_num += 1
                n_num_ok += n_lu == n_vrai
                etat += "" if n_lu == n_vrai else f" (numéro lu {n_lu[0]} ≠ {n_vrai[0] if n_vrai else '?'})"
        print(f"    {mmss(t_)} {d.get('equipe', '')} n° {d.get('marqueur', '?')} [{d.get('certitude')}] → {etat}")
    if CIBLES:
        PANIERS_CONNUS = [p for p in PANIERS_CONNUS if p in CIBLES]
        print(f"  (essai ciblé : bilan sur les {len(PANIERS_CONNUS)} paniers ciblés seulement)")
    manques = [p for p in PANIERS_CONNUS if all(abs(sec(p) - t_) > 8 for t_, _ in trouves)]
    faux = sum(1 for t_, _ in trouves if all(abs(sec(p) - t_) > 8 for p in PANIERS_CONNUS + INCERTAINS))
    print(f"  PANIERS TROUVÉS : {len(PANIERS_CONNUS) - len(manques)}/{len(PANIERS_CONNUS)} | manqués : {manques or 'aucun'}"
          f" | fausses alertes : {faux} | marqueur lu juste : {n_num_ok}/{n_num} des paniers où un numéro est donné")
if PANIERS_CONNUS:
    vus = sum(1 for d in reponses_seq if d["nature"] == "connu" and d.get("panier"))
    print(f"  paniers connus retrouvés : {vus}/{len(PANIERS_CONNUS)}")
faux = sum(1 for d in reponses_seq if d["nature"] == "hasard" and d.get("panier"))
print(f"  séquences au hasard vues comme « panier » : {faux}/{sum(1 for d in reponses_seq if d['nature'] == 'hasard')}"
      f" (sans vérité : à contrôler à l'œil dans la vidéo)")
try:
    gv.gemini_cache_summary()
except Exception:
    pass
json.dump({"video": VIDEO, "resolution": [w, h], "fps": fps, "duree_min": duree / 60, "images": reponses_img,
           "sequences": reponses_seq}, open(f"{SORTIE}/bilan_basket.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
with zipfile.ZipFile("/kaggle/working/test_basket.zip", "w") as z:
    for f in glob.glob(f"{SORTIE}/*"):
        z.write(f, os.path.basename(f))
print("✅ /kaggle/working/test_basket.zip (images annotées, réponses de Gemini, bilan)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.6/580.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.6/863.6 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/8

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.3 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.5.0 which is incompatible.
diffusers 0.36.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.5.0 which is incompatible.
gradio-client 1.14.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.5.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 2.5.0 which is incompatible.


  [GEMINI] Verrou installé : modèle unique gemini-3.8-flash + cache pour tous les appels du projet
VIDÉO : /kaggle/input/datasets/michaelziant/basketjeune/basketJeune.mp4
  1332x748 | 30.0 images/s | 31.2 min | portion testée 0→1872 s

══ 1. DÉTECTION (RF-DETR, comme au football) ══
  (sautée : déjà faite)

══ 2. GEMINI, IMAGE PAR IMAGE ══

══ 3. GEMINI, SÉQUENCES DE 20 s : PANIER MARQUÉ ? ══
  étape 1 : repérage des tirs (images toutes les 2 s)
  [GEMINI CACHE] mode=on | 11465 réponse(s) en cache (17 fichier(s) lu(s)) | sortie : /kaggle/working/test_basket/gemini_cache_basket.jsonl
  67 tir(s) repéré(s) : ['00:34 42 blanc', '01:04 vert', '01:32 3 blanc', '01:42 vert', '02:06 5 blanc', '02:20 14 blanc', '02:24 blanc et bleu', '02:28 3 blanc', '03:02 blanc', '05:06 31 vert', '05:10 15 vert', '05:36 3 blanc', '05:50 vert', '05:58 blanc', '06:22 31 vert', '06:42 3 vert', '06:48 31 vert', '07:28 14 blanc', '07:38 vert', '07:54 blanc', '08:06 31 vert', '08:26 3 blanc', '08:32 42 blanc', '10